# Serie A — App de Predicciones

**Requisito:** `modelo_italian.pkl` generado por `italian_league_train.ipynb`

```
Antes de la jornada     →  Celdas 1, 2, 3, 4, 5
Después de cada partido →  Celda 6
Ver resumen             →  Celda 7
```

**Umbral de uso:** Solo usar predicciones con confianza > 55% (marcadas `← USAR`)

In [1]:
# ─────────────────────────────────────────────
# CELDA 1 — Imports y carga del modelo
# Correr siempre al abrir el notebook
# ─────────────────────────────────────────────
import pandas as pd
import numpy as np
import joblib
import json

datos = joblib.load('modelo_italian.pkl')

model_v3        = datos['model_v3']
feature_cols_v3 = datos['feature_cols_v3']
le              = datos['le']
df              = datos['df']

print('Modelo cargado correctamente')
print(f'  Fecha entrenamiento:   {datos["fecha_entrenamiento"]}')
print(f'  Accuracy:              {datos["accuracy"]:.1%}')
print(f'  Accuracy alta conf:    {datos["accuracy_alta_confianza"]:.1%}')
print(f'  Features:              {len(feature_cols_v3)}')
print(f'\nEquipos disponibles:')
print(sorted(df['HomeTeam'].unique()))

Modelo cargado correctamente
  Fecha entrenamiento:   2026-04-07
  Accuracy:              51.1%
  Accuracy alta conf:    66.1%
  Features:              52

Equipos disponibles:
['Atalanta', 'Benevento', 'Bologna', 'Cagliari', 'Como', 'Cremonese', 'Crotone', 'Empoli', 'Fiorentina', 'Frosinone', 'Genoa', 'Inter', 'Juventus', 'Lazio', 'Lecce', 'Milan', 'Monza', 'Napoli', 'Parma', 'Pisa', 'Roma', 'Salernitana', 'Sampdoria', 'Sassuolo', 'Spezia', 'Torino', 'Udinese', 'Venezia', 'Verona']


In [2]:
# ─────────────────────────────────────────────
# CELDA 2 — Funciones del modelo
# Correr siempre al abrir el notebook
# ─────────────────────────────────────────────

def get_team_stats_v2(team, date, df, n_recent=5):
    home_all  = df[(df['HomeTeam'] == team) & (df['Date'] < date)]
    away_all  = df[(df['AwayTeam'] == team) & (df['Date'] < date)]
    all_games = pd.concat([home_all, away_all]).sort_values('Date')
    if len(all_games) < 3: return None
    current_season = df[df['Date'] < date]['season'].iloc[-1]
    season_games   = pd.concat([
        home_all[home_all['season'] == current_season],
        away_all[away_all['season'] == current_season]
    ]).sort_values('Date')
    recent = all_games.tail(n_recent)

    def calc_stats(games, team):
        if len(games) == 0: return None
        gf=gc=sot_f=sot_c=wins=draws=home_wins=home_games=0
        for _, r in games.iterrows():
            ih = r['HomeTeam'] == team
            gf    += r['FTHG'] if ih else r['FTAG']
            gc    += r['FTAG'] if ih else r['FTHG']
            sot_f += r['HST']  if ih else r['AST']
            sot_c += r['AST']  if ih else r['HST']
            won  = (ih and r['FTR']=='H') or (not ih and r['FTR']=='A')
            drew = r['FTR'] == 'D'
            if won:  wins  += 1
            if drew: draws += 1
            if ih:
                home_games += 1
                if r['FTR'] == 'H': home_wins += 1
        m = len(games)
        return {'gf_pg': gf/m, 'gc_pg': gc/m, 'dif_goles': (gf-gc)/m,
                'sot_f_pg': sot_f/m, 'sot_c_pg': sot_c/m,
                'win_rate': wins/m, 'draw_rate': draws/m,
                'home_wr': home_wins/home_games if home_games > 0 else wins/m}

    rs = calc_stats(recent, team)
    ss = calc_stats(season_games, team) if len(season_games) >= 3 else rs
    if rs is None: return None
    return {f'recent_{k}': rs[k] for k in rs} | {f'season_{k}': ss[k] if ss else rs[k] for k in rs}


def get_h2h_stats(home_team, away_team, date, df, n=10):
    h2h = df[
        ((df['HomeTeam']==home_team)&(df['AwayTeam']==away_team)) |
        ((df['HomeTeam']==away_team)&(df['AwayTeam']==home_team))
    ][lambda x: x['Date'] < date].tail(n)
    if len(h2h) < 3:
        return {'h2h_home_wr': 0.33, 'h2h_draw_rate': 0.33, 'h2h_away_wr': 0.33}
    hw = ((h2h['HomeTeam']==home_team)&(h2h['FTR']=='H')).sum() + \
         ((h2h['AwayTeam']==home_team)&(h2h['FTR']=='A')).sum()
    dr = (h2h['FTR']=='D').sum(); m = len(h2h)
    return {'h2h_home_wr': hw/m, 'h2h_draw_rate': dr/m, 'h2h_away_wr': (m-hw-dr)/m}


def get_tabla_posicion(team, date, df):
    season = df[df['Date'] < date]['season'].iloc[-1]
    ps     = df[(df['season']==season)&(df['Date']<date)]
    tabla  = []
    for eq in pd.concat([ps['HomeTeam'], ps['AwayTeam']]).unique():
        hp = ps[ps['HomeTeam']==eq]; ap = ps[ps['AwayTeam']==eq]
        tabla.append({'equipo': eq,
                      'pts': (hp['FTR']=='H').sum()*3+(hp['FTR']=='D').sum() +
                             (ap['FTR']=='A').sum()*3+(ap['FTR']=='D').sum(),
                      'gf': hp['FTHG'].sum()+ap['FTAG'].sum(),
                      'gc': hp['FTAG'].sum()+ap['FTHG'].sum(),
                      'pj': len(hp)+len(ap)})
    tdf = pd.DataFrame(tabla).sort_values('pts', ascending=False).reset_index(drop=True)
    tdf['pos'] = tdf.index + 1
    f = tdf[tdf['equipo']==team]
    if len(f)==0: return {'posicion': 10, 'pts_pg': 1.0, 'dif_goles_szn': 0, 'pct_posicion': 0.5}
    f = f.iloc[0]; pj = max(f['pj'],1)
    return {'posicion': f['pos'], 'pts_pg': f['pts']/pj,
            'dif_goles_szn': (f['gf']-f['gc'])/pj, 'pct_posicion': 1-(f['pos']/len(tdf))}


def predecir_partido_v3(home_team, away_team, fecha_str):
    """Predice H/D/A con probabilidades. Recomienda solo si confianza >55%."""
    fecha = pd.Timestamp(fecha_str)
    hs  = get_team_stats_v2(home_team, fecha, df)
    as_ = get_team_stats_v2(away_team, fecha, df)
    h2h = get_h2h_stats(home_team, away_team, fecha, df)
    ht2 = get_tabla_posicion(home_team, fecha, df)
    at2 = get_tabla_posicion(away_team, fecha, df)

    if hs is None or as_ is None:
        print(f'Sin historia suficiente: {home_team} o {away_team}')
        return None

    row = {}
    for k, v in hs.items():  row[f'h_{k}'] = v
    for k, v in as_.items(): row[f'a_{k}'] = v
    for k, v in h2h.items(): row[k] = v
    for k, v in ht2.items(): row[f'h_tabla_{k}'] = v
    for k, v in at2.items(): row[f'a_tabla_{k}'] = v
    row['dif_recent_wr']  = hs['recent_win_rate']  - as_['recent_win_rate']
    row['dif_season_wr']  = hs['season_win_rate']  - as_['season_win_rate']
    row['dif_recent_gf']  = hs['recent_gf_pg']     - as_['recent_gf_pg']
    row['dif_recent_gc']  = hs['recent_gc_pg']     - as_['recent_gc_pg']
    row['dif_recent_dif'] = hs['recent_dif_goles'] - as_['recent_dif_goles']
    row['dif_season_dif'] = hs['season_dif_goles'] - as_['season_dif_goles']
    row['home_advantage'] = hs['recent_home_wr']   - as_['recent_win_rate']
    row['dif_pts_pg']     = ht2['pts_pg']          - at2['pts_pg']
    row['dif_posicion']   = at2['posicion']        - ht2['posicion']

    proba     = model_v3.predict_proba(pd.DataFrame([row])[feature_cols_v3])[0]
    prob_A, prob_D, prob_H = proba[0], proba[1], proba[2]
    confianza = max(proba)
    pred      = le.inverse_transform([np.argmax(proba)])[0]
    usar      = confianza >= 0.55
    label     = {'H': f'{home_team} gana', 'D': 'Empate', 'A': f'{away_team} gana'}

    print(f"\n{'='*52}")
    print(f"  {home_team} vs {away_team}")
    print(f"{'='*52}")
    print(f"  Local gana     {home_team:<26} {prob_H:.1%}")
    print(f"  Empate                                      {prob_D:.1%}")
    print(f"  Visitante gana {away_team:<26} {prob_A:.1%}")
    print(f"{'─'*52}")
    print(f"  Prediccion:  {label[pred]}")
    print(f"  Confianza:   {confianza:.1%}  {'<- USAR' if usar else '<- IGNORAR'}")
    print(f"{'='*52}")

    return {'pred': pred, 'prob_H': prob_H, 'prob_D': prob_D,
            'prob_A': prob_A, 'confianza': confianza, 'usar': usar}


print('Funciones cargadas OK')

Funciones cargadas OK


In [3]:
# ─────────────────────────────────────────────
# CELDA 3 — Funciones de seguimiento
# Correr siempre al abrir el notebook
# ─────────────────────────────────────────────
predicciones = []

def registrar_prediccion(jornada, fecha, home, away, pred, prob_H, prob_D, prob_A, confianza):
    """Registra una prediccion ANTES del partido."""
    predicciones.append({
        'jornada': jornada, 'fecha': fecha, 'home': home, 'away': away,
        'prediccion': pred, 'prob_H': round(prob_H,4), 'prob_D': round(prob_D,4),
        'prob_A': round(prob_A,4), 'confianza': round(confianza,4),
        'resultado': None, 'correcto': None
    })
    print(f'Registrado: {home} vs {away} -> {pred} ({confianza:.1%})')


def registrar_resultado(home, away, resultado_real):
    """
    Registra el resultado DESPUES del partido.
    resultado_real: 'H' (local), 'D' (empate), 'A' (visitante)
    """
    for p in predicciones:
        if p['home'] == home and p['away'] == away:
            p['resultado'] = resultado_real
            p['correcto']  = p['prediccion'] == resultado_real
            print(f'{"ACERTADO" if p["correcto"] else "FALLADO"}: {home} vs {away} | '
                  f'Pred: {p["prediccion"]} | Real: {resultado_real}')
            return
    print(f'Partido no encontrado: {home} vs {away}')


def ver_seguimiento():
    """Muestra resumen de predicciones y accuracy acumulado."""
    if not predicciones:
        print('Sin predicciones registradas')
        return
    df_s      = pd.DataFrame(predicciones)
    jugados   = df_s['resultado'].notna().sum()
    acertados = df_s['correcto'].sum() if jugados > 0 else 0
    accuracy  = acertados / jugados if jugados > 0 else 0

    # Baseline Serie A: aprox 44% gana local
    baseline = 0.44

    print(f"\n{'='*52}")
    print(f"  RESUMEN — Serie A")
    print(f"{'='*52}")
    print(f"  Total predicciones:    {len(df_s)}")
    print(f"  Partidos jugados:      {jugados}")
    print(f"  Acertados:             {int(acertados)}")
    print(f"  Accuracy real:         {accuracy:.1%}")
    print(f"  Baseline (siempre H):  {baseline:.1%}")
    print(f"  Diferencia:            {(accuracy-baseline):+.1%}")
    print(f"\n{'='*52}")
    print(f"  DETALLE")
    print(f"{'='*52}")
    for _, r in df_s.iterrows():
        if r['resultado'] is not None:
            icon = 'OK' if r['correcto'] else 'XX'
            print(f"  [{icon}] J{r['jornada']} {r['home']} vs {r['away']}")
            print(f"       Pred: {r['prediccion']} ({r['confianza']:.1%}) | Real: {r['resultado']}")
        else:
            print(f"  [??] J{r['jornada']} {r['home']} vs {r['away']}")
            print(f"       Pred: {r['prediccion']} ({r['confianza']:.1%}) | Pendiente")

    if jugados > 0:
        df_j = df_s[df_s['resultado'].notna()].copy()
        df_j['cb'] = pd.cut(df_j['confianza'], bins=[0.50,0.55,0.60,0.65,1.0],
                            labels=['50-55%','55-60%','60-65%','>65%'])
        print(f"\n  Accuracy por confianza:")
        print(df_j.groupby('cb', observed=True).agg(
            partidos=('correcto','count'), accuracy=('correcto','mean')
        ).round(3).to_string())


def guardar_predicciones(archivo='predicciones.json'):
    """Guarda en disco para no perder al reiniciar el kernel."""
    datos = [{**p, 'correcto': bool(p['correcto']) if p['correcto'] is not None else None}
             for p in predicciones]
    with open(archivo, 'w') as f:
        json.dump(datos, f, indent=2)
    print(f'Guardado: {len(datos)} predicciones en {archivo}')


def cargar_predicciones(archivo='predicciones.json'):
    """Carga desde disco al reiniciar el kernel."""
    global predicciones
    try:
        with open(archivo, 'r') as f:
            predicciones = json.load(f)
        print(f'Cargado: {len(predicciones)} predicciones')
    except FileNotFoundError:
        print('No hay predicciones guardadas — lista vacia')


print('Funciones de seguimiento cargadas OK')

Funciones de seguimiento cargadas OK


In [4]:
# ─────────────────────────────────────────────
# CELDA 4 — Predecir jornada
# Editar: partidos_jornada y fecha_jornada
#
# Nombres exactos como aparecen en el dataset
# (ver lista de equipos en Celda 1)
# ─────────────────────────────────────────────

# ── EDITAR AQUI ──────────────────────────────
partidos_jornada = [
    ('Roma',        'Pisa'),          # Viernes  10 abr
    ('Cagliari',    'Cremonese'),      # Sábado   11 abr
    ('Torino',      'Verona'),         # Sábado   11 abr
    ('Milan',       'Udinese'),        # Sábado   11 abr
    ('Atalanta',    'Juventus'),       # Sábado   11 abr
    ('Genoa',       'Sassuolo'),       # Domingo  12 abr
    ('Parma',       'Napoli'),         # Domingo  12 abr
    ('Bologna',     'Lecce'),          # Domingo  12 abr
    ('Como',        'Inter'),          # Domingo  12 abr
    ('Fiorentina',  'Lazio'),          # Lunes    13 abr
]

fecha_jornada = '2026-04-10'
# ─────────────────────────────────────────────

resultados = {}
for home, away in partidos_jornada:
    res = predecir_partido_v3(home, away, fecha_jornada)
    if res:
        resultados[f'{home} vs {away}'] = res


  Roma vs Pisa
  Local gana     Roma                       65.1%
  Empate                                      25.6%
  Visitante gana Pisa                       9.3%
────────────────────────────────────────────────────
  Prediccion:  Roma gana
  Confianza:   65.1%  <- USAR

  Cagliari vs Cremonese
  Local gana     Cagliari                   38.5%
  Empate                                      34.1%
  Visitante gana Cremonese                  27.5%
────────────────────────────────────────────────────
  Prediccion:  Cagliari gana
  Confianza:   38.5%  <- IGNORAR

  Torino vs Verona
  Local gana     Torino                     46.1%
  Empate                                      34.5%
  Visitante gana Verona                     19.4%
────────────────────────────────────────────────────
  Prediccion:  Torino gana
  Confianza:   46.1%  <- IGNORAR

  Milan vs Udinese
  Local gana     Milan                      44.2%
  Empate                                      38.5%
  Visitante gana Udinese  

In [ ]:
# ─────────────────────────────────────────────
# CELDA 5 — Registrar predicciones a usar
# Solo registrar las marcadas con <- USAR
# ─────────────────────────────────────────────
cargar_predicciones()

# ── EDITAR AQUI — solo confianza >55% ────────
# Ejemplo — reemplazar con los resultados reales de la Celda 4:
#
# registrar_prediccion(
#     jornada=32, fecha='2026-04-10',
#     home='Roma', away='Pisa',
#     pred='H', prob_H=0.651, prob_D=0.256, prob_A=0.093,
#     confianza=0.651
# )

# registrar_prediccion(
#     jornada=32, fecha='2026-04-12',
#     home='Bologna', away='Lecce',
#     pred='H', prob_H=0.609, prob_D=0.262, prob_A=0.129,
#     confianza=0.609
# )
# ─────────────────────────────────────────────

guardar_predicciones()
print()
ver_seguimiento()

Cargado: 0 predicciones
Registrado: Roma vs Pisa -> H (65.1%)
Registrado: Bologna vs Lecce -> H (60.9%)
Guardado: 2 predicciones en predicciones.json


  RESUMEN — Serie A
  Total predicciones:    2
  Partidos jugados:      0
  Acertados:             0
  Accuracy real:         0.0%
  Baseline (siempre H):  44.0%
  Diferencia:            -44.0%

  DETALLE
  [??] J32 Roma vs Pisa
       Pred: H (65.1%) | Pendiente
  [??] J32 Bologna vs Lecce
       Pred: H (60.9%) | Pendiente


In [6]:
# ─────────────────────────────────────────────
# CELDA 6 — Registrar resultados reales
# Correr DESPUES de cada partido
#
# H = gana local    D = empate    A = gana visitante
# ─────────────────────────────────────────────
# cargar_predicciones()

# # ── EDITAR AQUI — cambiar por resultado real ─
# registrar_resultado('Inter',    'AC Milan', 'H')  # <- cambiar despues del partido
# registrar_resultado('Juventus', 'Napoli',   'D')  # <- cambiar despues del partido
# # ─────────────────────────────────────────────

# guardar_predicciones()

In [7]:
# ─────────────────────────────────────────────
# CELDA 7 — Ver resumen completo
# Correr en cualquier momento
# ─────────────────────────────────────────────
cargar_predicciones()
ver_seguimiento()

Cargado: 0 predicciones
Sin predicciones registradas
